# 🛸 Complete Unsupervised Anomaly Detection: All Models Benchmark (Epochs=15)

Evaluates **all models** (Pointwise Classical + Deep Sequence Autoencoders) across the complete feature progression:
1. **Baseline 8 Features** (Pure Kinematics without Yaw Accel & PE)
2. **Baseline 10 Features** (Kinematics + Prediction Error + Yaw Acceleration)
3. **Noise Texture 13 Features** (10 Baseline + PE Autocorrelation + Position Noise Std + Speed Spectral Entropy)
4. **Baseline + Cross-Correlation 13 Features** (10 Baseline + Speed-Turn Corr + Accel-Turn Corr + Vert-Speed Corr)
5. **Cross-Correlation 16 Features** (13 Noise Texture + 3 Cross-Correlations)

### Models Evaluated (12 Total):
- **Pointwise Models (8)**: Isolation Forest, GMM, Mahalanobis, One-Class SVM, PCA ($Q + T^2$), K-Means, DBSCAN, KNN
- **Deep Sequence Autoencoders (4)**: Dense AE, GRU AE, TCN AE, CNN-GRU AE

### Datasets Evaluated (7 Classes):
- `Normal DJI` (TNR / Clean baseline), `Real ESP32`, `Sim Baseline`, `Sim Easy`, `Sim Medium`, `Sim Hard`, `Sim Geometry`.

## 1. Setup Working Directory & Environment

In [ ]:
from pathlib import Path
import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from presets.run_unsupervised_pipeline import FEATURE_SETS, run_experiment
from implement.utils.helper import get_output_dir

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Operating Directory: {os.getcwd()}")
print(f"Compute Device: {device} | PyTorch: {torch.__version__}")
print(f"Available Feature Presets: {list(FEATURE_SETS.keys())}")

## 2. Experiment 1: Pure Kinematic Core (8 Features — All Models)
- Features: `height`, `ground_speed`, `vertical_speed`, `acceleration`, `turn_rate`, `path_curvature`, `heading_speed_consistency`, `motion_smoothness`

In [ ]:
df_exp1_b8 = run_experiment(
    exp_name="all_models_baseline_8",
    feature_list=FEATURE_SETS["baseline_8"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 3. Experiment 2: Standard Baseline (10 Features — All Models)
- Features: 8 Kinematics + `prediction_error`, `yaw_acceleration`

In [ ]:
df_exp2_b10 = run_experiment(
    exp_name="all_models_baseline_10",
    feature_list=FEATURE_SETS["baseline_10"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 4. Experiment 3: Noise Texture Extended (13 Features — All Models)
- Features: 10 Baseline + `prediction_error_autocorrelation`, `position_residual_std`, `speed_spectral_entropy`

In [ ]:
df_exp3_nt13 = run_experiment(
    exp_name="all_models_noise_texture_13",
    feature_list=FEATURE_SETS["noise_texture_13"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 5. Experiment 4: Baseline + Cross-Correlation (13 Features — All Models)
- Features: 10 Baseline + `corr_speed_turn`, `corr_accel_turn`, `corr_vert_speed`

In [ ]:
df_exp4_bc13 = run_experiment(
    exp_name="all_models_baseline_corr_13",
    feature_list=FEATURE_SETS["baseline_corr_13"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 6. Experiment 5: Full Cross-Correlation & Noise Texture (16 Features — All Models)
- Features: 13 Noise Texture + `corr_speed_turn`, `corr_accel_turn`, `corr_vert_speed`

In [ ]:
df_exp5_cc16 = run_experiment(
    exp_name="all_models_correlation_16",
    feature_list=FEATURE_SETS["correlation_16"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 7. Comprehensive Multi-Experiment Summary & Aggregate Comparison

In [ ]:
experiments = [
    ("Baseline 8", "all_models_baseline_8"),
    ("Baseline 10", "all_models_baseline_10"),
    ("Noise Texture 13", "all_models_noise_texture_13"),
    ("Baseline + Corr 13", "all_models_baseline_corr_13"),
    ("Cross-Correlation 16", "all_models_correlation_16")
]

agg_frames = []
for label, exp in experiments:
    p = get_output_dir() / "pipeline_experiments" / exp / f"{exp}_aggregate.csv"
    if p.exists():
        df = pd.read_csv(p)
        df.insert(0, "Feature Set", label)
        agg_frames.append(df)

if agg_frames:
    full_agg_df = pd.concat(agg_frames, ignore_index=True)
    summary_save_path = get_output_dir() / "all_models_feature_progression_aggregate_summary.csv"
    full_agg_df.to_csv(summary_save_path, index=False)
    print(f"✅ Aggregated summary across all feature progressions saved to: {summary_save_path}")
    display(full_agg_df)
else:
    print("Run the experiment cells above to produce aggregate summaries.")